# Patrón de Comportamiento: Memento

## Introducción
El patrón Memento permite capturar y restaurar el estado interno de un objeto sin violar su encapsulamiento.

## Objetivos
- Comprender cómo guardar y restaurar el estado de un objeto.
- Identificar cuándo es útil el patrón Memento.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: Editor de texto con deshacer/rehacer**
Un editor de texto puede guardar el estado del documento para permitir deshacer y rehacer cambios.

**¿Dónde se usa en proyectos reales?**
En editores, juegos, sistemas de configuración, etc.

## Sin patrón Memento (forma errónea)
El estado se guarda manualmente, exponiendo detalles internos.

In [1]:
class Documento:
    def __init__(self):
        self.texto = ''
    def escribir(self, txt):
        self.texto += txt
    def mostrar(self):
        print(self.texto)
    def guardar_estado(self):
        return self.texto
    def restaurar_estado(self, estado):
        self.texto = estado

doc = Documento()
doc.escribir('Hola')
estado = doc.guardar_estado()
doc.escribir(' Mundo')
doc.restaurar_estado(estado)
doc.mostrar()

Hola


## Con patrón Memento (forma correcta)
El estado se encapsula en un objeto memento.

In [2]:
class Memento:
    def __init__(self, estado):
        self._estado = estado
    def get_estado(self):
        return self._estado

class Documento:
    def __init__(self):
        self.texto = ''
    def escribir(self, txt):
        self.texto += txt
    def mostrar(self):
        print(self.texto)
    def crear_memento(self):
        return Memento(self.texto)
    def restaurar(self, memento):
        self.texto = memento.get_estado()

class Historial:
    def __init__(self):
        self.mementos = []
    def guardar(self, memento):
        self.mementos.append(memento)
    def deshacer(self):
        if self.mementos:
            return self.mementos.pop()

doc = Documento()
hist = Historial()
doc.escribir('Hola')
hist.guardar(doc.crear_memento())
doc.escribir(' Mundo')
doc.mostrar()
doc.restaurar(hist.deshacer())
doc.mostrar()

Hola Mundo
Hola


## UML del patrón Memento
```plantuml
@startuml
class Documento {
    + escribir(txt)
    + mostrar()
    + crear_memento()
    + restaurar(memento)
}
class Memento {
    + get_estado()
}
class Historial {
    + guardar(memento)
    + deshacer()
}
Documento --> Memento
Historial --> Memento
@enduml
```

## Otro ejemplo de la vida real: Rollback de configuración en un pipeline de CI/CD
**Contexto:** antes de aplicar un cambio de configuración a producción (ej. escalar réplicas, cambiar de versión), un sistema de despliegue debería poder revertir automáticamente al estado anterior si el nuevo despliegue falla. Guardar ese estado anterior sin exponer los detalles internos del pipeline es exactamente el problema de Memento.

### Sin patrón (forma errónea)
Se guarda una referencia directa a la configuración en vez de una copia — si el objeto de configuración es mutable, "el respaldo" puede terminar apuntando al mismo objeto que se quiere reemplazar.

In [3]:
class PipelineCICD:
    def __init__(self):
        self.configuracion = {'replicas': 2, 'version': 'v1.0'}
    def desplegar(self, cambios):
        self.configuracion.update(cambios)  # mutación in-place del mismo diccionario
        print(f'Desplegando con config: {self.configuracion}')

pipeline = PipelineCICD()
config_anterior = pipeline.configuracion  # esto NO es una copia, es la misma referencia
pipeline.desplegar({'replicas': 5, 'version': 'v2.0-beta'})
print('Config "de respaldo" (debería seguir en v1.0):', config_anterior)  # bug: también cambió

Desplegando con config: {'replicas': 5, 'version': 'v2.0-beta'}
Config "de respaldo" (debería seguir en v1.0): {'replicas': 5, 'version': 'v2.0-beta'}


### Con patrón (forma correcta)
`MementoConfiguracion` guarda una copia profunda e inmutable del estado. `HistorialDespliegues` administra los snapshots sin conocer los detalles internos de `PipelineCICD`, y el rollback siempre restaura el estado exacto de antes.

In [4]:
import copy

class MementoConfiguracion:
    def __init__(self, configuracion):
        self._configuracion = copy.deepcopy(configuracion)
    def obtener(self):
        return self._configuracion


class PipelineCICD:
    def __init__(self):
        self.configuracion = {'replicas': 2, 'version': 'v1.0'}
    def crear_snapshot(self):
        return MementoConfiguracion(self.configuracion)
    def desplegar(self, nueva_config):
        print(f'Desplegando con config: {nueva_config}')
        self.configuracion = nueva_config
    def rollback(self, memento):
        self.configuracion = memento.obtener()
        print(f'Rollback aplicado, config restaurada: {self.configuracion}')


class HistorialDespliegues:
    def __init__(self):
        self.snapshots = []
    def guardar(self, snapshot):
        self.snapshots.append(snapshot)
    def ultimo(self):
        return self.snapshots.pop() if self.snapshots else None


pipeline = PipelineCICD()
historial = HistorialDespliegues()

historial.guardar(pipeline.crear_snapshot())
pipeline.desplegar({'replicas': 5, 'version': 'v2.0-beta'})
print('Config actual:', pipeline.configuracion)

# El despliegue de v2.0-beta falló: se revierte al snapshot anterior
pipeline.rollback(historial.ultimo())

Desplegando con config: {'replicas': 5, 'version': 'v2.0-beta'}
Config actual: {'replicas': 5, 'version': 'v2.0-beta'}
Rollback aplicado, config restaurada: {'replicas': 2, 'version': 'v1.0'}


### UML del ejemplo de rollback de CI/CD
```plantuml
@startuml
class PipelineCICD {
    + configuracion
    + crear_snapshot()
    + desplegar(nueva_config)
    + rollback(memento)
}
class MementoConfiguracion {
    - _configuracion
    + obtener()
}
class HistorialDespliegues {
    - snapshots: list
    + guardar(snapshot)
    + ultimo()
}
PipelineCICD --> MementoConfiguracion
HistorialDespliegues --> MementoConfiguracion
@enduml
```

### ¿Dónde más se usa Memento?
- **Rollback de infraestructura/CI-CD:** exactamente este ejemplo — Kubernetes, Terraform o Helm guardan el estado anterior para poder revertir un despliegue fallido.
- **Editores con deshacer/rehacer:** el ejemplo con el que abre este notebook — guardar snapshots del documento para poder deshacer cambios.
- **Videojuegos:** guardar el estado del jugador (posición, vida, inventario) en puntos de control (checkpoints) para poder recargar la partida.
- **Bases de datos transaccionales:** un `SAVEPOINT` en SQL es, conceptualmente, un memento al que se puede volver con `ROLLBACK TO`.
- **Control de versiones:** cada commit de Git es, en cierto sentido, un memento del estado completo del repositorio en ese momento.

**Ejercicio de reflexión:** ¿por qué `MementoConfiguracion` usa `copy.deepcopy` en vez de guardar la referencia directa al diccionario de configuración? Relaciona tu respuesta con el bug que viste en la versión "sin patrón".

## Actividad
Crea un sistema de juego donde puedas guardar y restaurar el estado del jugador usando Memento.

---
## Explicación de conceptos clave
- **Encapsulamiento:** El estado se guarda sin exponer detalles internos.
- **Deshacer/rehacer:** Permite restaurar estados previos fácilmente.
- **Aplicación en la vida real:** Útil en editores, juegos y sistemas de configuración.

## Conclusión
El patrón Memento es ideal para guardar y restaurar el estado de objetos de manera segura y encapsulada.